# Demo · Permessi per documento (ACL)

**Use case** (route `/demo/acl` della chat): la stessa domanda dà risultati diversi a seconda di **chi** la pone. Foundry IQ ingerisce le ACL ADLS Gen2 di ogni documento e, con il token dell'utente in `x-ms-query-source-authorization`, restituisce solo i documenti che l'utente può leggere.

Il notebook legge l'ambiente azd selezionato oppure gli output ARM e **non modifica** gruppi o ACL: confronta le appartenenze Entra e le ACL di origine con ciò che Foundry IQ restituisce.

| Passo | Mostra |
| --- | --- |
| 1 | Chi sono: utente connesso e appartenenza ai gruppi *Tecnici* e *Direzione*. |
| 2 | ACL di origine: quali gruppi leggono ogni cartella. |
| 3 | Knowledge source: ingestione dei permessi. |
| 4 | Confronto: senza identità utente e con il token dell'utente. |

In [ ]:
CONFIG_SOURCE = "azd"             # "azd" oppure "arm"
AZD_ENVIRONMENT_NAME = ""          # vuoto = ambiente azd attualmente selezionato
SUBSCRIPTION_ID = ""               # opzionale; con azd deve coincidere con l'ambiente
ENVIRONMENT_NAME = "wpc-demo"      # usato solo per il deployment ARM
DEPLOYMENT_NAME = f"wpc-foundryiq-{ENVIRONMENT_NAME}"
SEARCH_API_VERSION = "2026-11-01-preview"
ACL_KB_NAME = ""                   # vuoto = nome dell'ambiente o wpc-acl-kb
ACL_KS = ""                        # vuoto = unica knowledge source della KB ACL
GROUPS = {"Tecnici": "WPC FoundryIQ ACL - Tecnici", "Direzione": "WPC FoundryIQ ACL - Direzione"}
GROUP_IDS = {"Tecnici": "", "Direzione": ""}  # opzionali; hanno precedenza sugli ID nell'ambiente

# Stesse domande della chat: cartella del documento atteso e gruppi autorizzati a leggerlo.
SCENARI = [
    {"titolo": "Procedura richiesta ricambi · SLA P1", "cartella": "pubblico", "gruppi": ["Tecnici", "Direzione"],
     "domanda": "Qual è il codice della procedura di richiesta ricambi e lo SLA per la priorità P1?"},
    {"titolo": "Ven Model 12 · modalità service", "cartella": "manutenzione-riservata", "gruppi": ["Tecnici"],
     "domanda": "Qual è il codice della modalità service del Ven Systems Model 12?"},
    {"titolo": "Accordi quadro 2027 · Delta", "cartella": "direzione-riservata", "gruppi": ["Direzione"],
     "domanda": "Qual è lo sconto concordato con Delta Systems negli accordi quadro 2027?"},
]

In [ ]:
import json
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "infra" / "main.bicep").is_file()), None)
if ROOT is None:
    raise RuntimeError("Apri il notebook dentro il repository: servono infra/, data/, agents/ e le app.")
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
from notebook_env import load_azd_env
AZ = shutil.which("az")
if AZ is None:
    raise RuntimeError("Azure CLI non trovata nel PATH: installala, esegui 'az login' e riavvia il kernel.")


def az(*args, output=True, check=True):
    """Esegue Azure CLI. Con output=True restituisce il JSON decodificato; con check=False None in caso di errore."""
    command = [AZ, *args, "--only-show-errors"] + (["--output", "json"] if output else [])
    proc = subprocess.run(command, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        if check:
            raise RuntimeError(f"az {' '.join(args[:3])} fallito: {proc.stderr.strip()[-2000:]}")
        return None
    text = proc.stdout.strip()
    return (json.loads(text) if text else None) if output else text


def load_azd_values(environment_name):
    return load_azd_env(environment_name, root=ROOT, override=True)


def load_outputs(deployment_name, subscription_id):
    deployment = az("deployment", "sub", "show", "--name", deployment_name, "--subscription", subscription_id)
    if not deployment:
        raise RuntimeError(f"Deployment ARM '{deployment_name}' non trovato nella subscription selezionata.")
    state = deployment["properties"].get("provisioningState")
    if state != "Succeeded":
        raise RuntimeError(f"Deployment '{deployment_name}' in stato {state}: correggi e riesegui 01-infrastruttura.ipynb.")
    return {name.upper(): item["value"] for name, item in (deployment["properties"].get("outputs") or {}).items()}


def load_configuration(required):
    if CONFIG_SOURCE == "azd":
        out = load_azd_values(AZD_ENVIRONMENT_NAME)
        selected = out.get("AZURE_SUBSCRIPTION_ID")
        if not isinstance(selected, str) or not selected.strip():
            raise RuntimeError("AZURE_SUBSCRIPTION_ID mancante nell'ambiente azd selezionato.")
        if SUBSCRIPTION_ID and SUBSCRIPTION_ID.casefold() != selected.casefold():
            raise RuntimeError("SUBSCRIPTION_ID non corrisponde alla subscription dell'ambiente azd.")
        account = az("account", "show", "--subscription", selected)
        if account["id"].casefold() != selected.casefold():
            raise RuntimeError("Azure CLI ha restituito una subscription diversa da quella dell'ambiente azd.")
        source = f"ambiente azd {out.get('AZURE_ENV_NAME') or AZD_ENVIRONMENT_NAME or '(corrente)'}"
    elif CONFIG_SOURCE == "arm":
        account = az("account", "show", *(["--subscription", SUBSCRIPTION_ID] if SUBSCRIPTION_ID else []))
        out = load_outputs(DEPLOYMENT_NAME, account["id"])
        source = f"deployment ARM {DEPLOYMENT_NAME}"
    else:
        raise ValueError("CONFIG_SOURCE deve essere 'azd' oppure 'arm'.")
    missing = [key for key in ["AZURE_TENANT_ID", "AZURE_RESOURCE_GROUP", *required]
               if not isinstance(out.get(key), str) or not out[key].strip()]
    if missing:
        raise RuntimeError(f"Valori mancanti in {source}: {', '.join(missing)}.")
    if out["AZURE_TENANT_ID"].casefold() != account["tenantId"].casefold():
        raise RuntimeError("Il tenant della configurazione non corrisponde alla subscription selezionata in Azure CLI.")
    return out, account, source


out, account, source = load_configuration(["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT", "FOUNDRYIQ_ACL_DFS_ENDPOINT", "FOUNDRYIQ_ACL_CONTAINER"])
SUBSCRIPTION_ID, TENANT_ID = account["id"], account["tenantId"]
RESOURCE_GROUP = out["AZURE_RESOURCE_GROUP"]
print(f"Subscription: {account['name']} ({SUBSCRIPTION_ID})")
print(f"Configurazione: {source} -> resource group {RESOURCE_GROUP}")

import pandas as pd
import requests
from azure.identity import AzureCliCredential
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)
credential = AzureCliCredential(subscription=SUBSCRIPTION_ID)
SEARCH_SCOPE = "https://search.azure.com/.default"
checks = []


def record(test, status, detail=""):
    """Esito di un controllo: PASS, WARN (non bloccante), FAIL o SKIP."""
    checks.append({"controllo": test, "esito": status, "dettaglio": detail})
    print(f"[{status}] {test}" + (f" - {detail}" if detail else ""))

from urllib.parse import quote, unquote, urlsplit

SEARCH = out["FOUNDRYIQ_DEMO_SEARCH_ENDPOINT"].rstrip("/")
ACL_KB_NAME = ACL_KB_NAME or out.get("FOUNDRYIQ_ACL_KB_NAME") or "wpc-acl-kb"


def search_token():
    return credential.get_token(SEARCH_SCOPE).token


def search_get(path):
    response = requests.get(f"{SEARCH}/{path}?api-version={SEARCH_API_VERSION}", timeout=60,
                            headers={"Authorization": f"Bearer {search_token()}"})
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()


def retrieve(kb_name, body, user_token=None):
    """La credenziale Azure CLI autorizza Search; il token sorgente opzionale applica le ACL."""
    headers = {"Authorization": f"Bearer {search_token()}"}
    if user_token:
        headers["x-ms-query-source-authorization"] = user_token
    started = time.perf_counter()
    response = requests.post(f"{SEARCH}/knowledgebases/{quote(kb_name, safe='')}/retrieve?api-version={SEARCH_API_VERSION}",
                             headers=headers, json=body, timeout=180)
    elapsed = round((time.perf_counter() - started) * 1000)
    if not response.ok:
        raise RuntimeError(f"Retrieve {kb_name}: HTTP {response.status_code} {response.text[:600]}")
    result = response.json()
    result["_elapsed_ms"] = elapsed
    return result

CONTAINER = out["FOUNDRYIQ_ACL_CONTAINER"]


def reference_path(ref):
    url = ref.get("blobUrl") or ref.get("url") or (ref.get("sourceData") or {}).get("url") or ""
    parsed = urlsplit(url)
    dfs_host = urlsplit(out["FOUNDRYIQ_ACL_DFS_ENDPOINT"]).hostname
    blob_host = dfs_host.replace(".dfs.", ".blob.", 1) if dfs_host else None
    marker = f"/{CONTAINER}/"
    path = unquote(parsed.path)
    if parsed.scheme == "https" and parsed.hostname in {dfs_host, blob_host} and path.startswith(marker):
        return path[len(marker):] or None
    return None


def source_paths(result):
    """Percorsi verificabili nel container ACL dell'account configurato."""
    return sorted({path for ref in result.get("references") or [] if (path := reference_path(ref))})


def evaluate_access(result, scenario, membership, scenarios):
    paths = source_paths(result)
    unresolved = sum(1 for ref in result.get("references") or [] if not reference_path(ref))
    allowed = any(membership[group] for group in scenario["gruppi"])
    found = any(path.startswith(scenario["cartella"] + "/") for path in paths)
    forbidden = [path for path in paths if any(path.startswith(s["cartella"] + "/") and
                 not any(membership[group] for group in s["gruppi"]) for s in scenarios)]
    unknown = [path for path in paths if not any(path.startswith(s["cartella"] + "/") for s in scenarios)]
    if forbidden:
        status = "FAIL"
    elif unresolved or unknown:
        status = "WARN"
    elif allowed and not found:
        status = "FAIL"
    elif not allowed:
        status = "WARN"
    else:
        status = "PASS"
    return {"paths": paths, "allowed": allowed, "found": found, "forbidden": forbidden,
            "unverifiable": unresolved + len(unknown), "status": status}

## 1 · Chi sono

In [ ]:
graph_account = az("account", "show")
if graph_account["tenantId"].casefold() != TENANT_ID.casefold():
    raise RuntimeError("Il tenant predefinito di Azure CLI non coincide con quello della demo: accedi al tenant corretto prima di usare az ad.")
me = az("ad", "signed-in-user", "show")
group_settings = {"Tecnici": "FOUNDRYIQ_ACL_GROUP_TECNICI_ID", "Direzione": "FOUNDRYIQ_ACL_GROUP_DIREZIONE_ID"}
group_ids, group_names, membership = {}, {}, {}
for label, name in GROUPS.items():
    configured_id = GROUP_IDS.get(label) or out.get(group_settings[label])
    if configured_id:
        group = az("ad", "group", "show", "--group", configured_id)
    else:
        matches = [g for g in az("ad", "group", "list", "--display-name", name) if g["displayName"] == name]
        if len(matches) > 1:
            raise RuntimeError(f"Più gruppi Entra hanno il nome '{name}': imposta GROUP_IDS['{label}'] o {group_settings[label]} nell'ambiente azd.")
        group = matches[0] if matches else None
    group_ids[label] = group["id"] if group else None
    group_names[label] = group.get("displayName", name) if group else name
    membership[label] = bool(group) and az("ad", "group", "member", "check", "--group", group["id"], "--member-id", me["id"])["value"]
display(pd.DataFrame([{"gruppo": f"{label} ({group_names[label]})", "object id": group_ids[label] or "non trovato",
                       "sei membro": "sì" if membership[label] else "no"} for label in GROUPS]))
print(f"Utente: {me['displayName']} <{me['userPrincipalName']}>")
record("Gruppi demo presenti", "PASS" if all(group_ids.values()) else "FAIL", "configura i gruppi ACL se mancano")
if not all(group_ids.values()):
    raise RuntimeError("Gruppi ACL mancanti: impossibile determinare le aspettative di accesso.")

## 2 · ACL di origine su ADLS Gen2

In [ ]:
from azure.storage.filedatalake import DataLakeServiceClient

names_by_id = {v: k for k, v in group_ids.items() if v}
acl_rows = []
with DataLakeServiceClient(out["FOUNDRYIQ_ACL_DFS_ENDPOINT"], credential=credential) as service:
    fs = service.get_file_system_client(CONTAINER)
    for scenario in SCENARI:
        acl = fs.get_directory_client(scenario["cartella"]).get_access_control()["acl"]
        readers = sorted(names_by_id.get(e.split(":")[1], e.split(":")[1]) for e in acl.split(",")
                         if e.startswith("group:") and e.split(":")[1] and "r" in e.split(":")[2])
        files = [p.name.split("/")[-1] for p in fs.get_paths(scenario["cartella"]) if not p.is_directory]
        acl_rows.append({"cartella": scenario["cartella"], "gruppi con lettura": ", ".join(readers), "file": ", ".join(files)})
        record(f"ACL origine · {scenario['cartella']}", "PASS" if sorted(scenario["gruppi"]) == readers else "FAIL",
               f"attesi {', '.join(scenario['gruppi'])}")
display(pd.DataFrame(acl_rows))

## 3 · Ingestione dei permessi

In [ ]:
kb = search_get(f"knowledgebases/{quote(ACL_KB_NAME, safe='')}")
if not kb:
    raise RuntimeError(f"Knowledge base ACL '{ACL_KB_NAME}' non trovata.")
sources = [s.get("name") for s in kb.get("knowledgeSources") or [] if s.get("name")]
if not ACL_KS:
    if len(sources) != 1:
        raise RuntimeError(f"La KB ACL ha {len(sources)} knowledge source: imposta ACL_KS esplicitamente.")
    ACL_KS = sources[0]
if ACL_KS not in sources:
    raise RuntimeError(f"Knowledge source '{ACL_KS}' non associata alla KB '{ACL_KB_NAME}'.")
reasoning = (kb.get("retrievalReasoningEffort") or {}).get("kind")
if reasoning != "minimal":
    raise RuntimeError(f"La demo ACL richiede reasoning minimal, configurato: {reasoning}.")
print(f"KB: {ACL_KB_NAME} | knowledge source: {ACL_KS} | reasoning: {reasoning}")
ks = search_get(f"knowledgesources/{quote(ACL_KS, safe='')}") or {}
options = ((ks.get("azureBlobParameters") or {}).get("ingestionParameters") or {}).get("ingestionPermissionOptions") or []
record("Knowledge source con ACL", "PASS" if {"userIds", "groupIds"} <= set(options) else "FAIL",
       f"ingestionPermissionOptions = {options}")

## 4 · Stessa domanda, identità diverse

- **Senza token utente della sorgente**: la credenziale Azure CLI autorizza Search, ma non viene inviato l'header per le ACL; i documenti protetti non devono comparire.
- **Con il tuo token**: compaiono solo i documenti delle cartelle che i tuoi gruppi possono leggere.

La KB usa reasoning `minimal`, quindi riceve un intento di ricerca esplicito. Un documento assente dai risultati non dimostra, da solo, che le ACL ne abbiano impedito la restituzione.

In [ ]:
user_token = search_token()
comparison = []
for scenario in SCENARI:
    body = {"intents": [{"type": "semantic", "search": scenario["domanda"]}], "includeActivity": True}
    anonymous = retrieve(ACL_KB_NAME, body)
    delegated = retrieve(ACL_KB_NAME, body, user_token)
    access = evaluate_access(delegated, scenario, membership, SCENARI)
    comparison.append({"scenario": scenario["titolo"], "autorizzati": ", ".join(scenario["gruppi"]),
                       "dovresti vederlo": "sì" if access["allowed"] else "no",
                       "senza identità": len(anonymous.get("references") or []),
                       "con il tuo token": ", ".join(access["paths"]) or "nessun documento verificabile"})
    record(f"Senza identità · {scenario['cartella']}", "PASS" if not anonymous.get("references") else "FAIL",
           f"{len(anonymous.get('references') or [])} riferimenti")
    record(f"Con token · {scenario['cartella']}", access["status"],
           f"atteso={'visibile' if access['allowed'] else 'nascosto'}; trovato={access['found']}; "
           f"vietati={access['forbidden'] or 'nessuno'}; non verificabili={access['unverifiable']}" +
           ("; assenza non prova un diniego" if not access["allowed"] and not access["found"] else ""))
display(pd.DataFrame(comparison))

Per vedere cambiare il risultato, aggiungi o rimuovi l'utente da un gruppo nel portale Entra, attendi la propagazione e riesegui dalla sezione 1. Se necessario, ottieni un nuovo token. Nella chat web la stessa differenza si vede con la modalità **Con ACL / Senza ACL**.

In [ ]:
credential.close()
display(pd.DataFrame(checks))
failed = [c["controllo"] for c in checks if c["esito"] == "FAIL"]
if failed:
    raise AssertionError("Controlli falliti: " + ", ".join(failed))
print("Demo ACL completata: verifica gli eventuali avvisi; l'assenza di riferimenti non dimostra un diniego.")